In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import gc
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
INPUT_FORECAST = "Stage1_Forecast_EXP_CO2_FORCED.csv"  # The Stage I Output
DATA_DIR = "data/processed/merged_5var_dataset.parquet" # The Master History
OUTPUT_DIR = "outputs/stage2_biology"
os.makedirs(OUTPUT_DIR, exist_ok=True)

REGION_BOX = {
    "name": "Great Barrier Reef (Regional Stats)",
    "lat_min": -24.0, "lat_max": -10.0,
    "lon_min": 142.0, "lon_max": 154.0
}

# --- STYLE SETTINGS ---
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.4)
colors = {'chl': '#2ca02c', 'poc': '#8c564b', 'stress': '#d62728'}

# ==========================================
# 1. PREPARE TRAINING DATA (History)
# ==========================================
def load_training_history(folder_path, box):
    print("--- 1. Extraction: Learning Biological Rules (2002-2025) ---")
    
    # We need to extract ALL 5 variables now (Inputs + Targets)
    files = sorted(glob.glob(os.path.join(folder_path, "*.parquet")))
    history_data = []
    
    # Vars to extract
    # Inputs: SST, PAR, Kd490
    # Targets: Chlorophyll, POC
    target_vars = ['sst', 'par', 'kd_490', 'chlor_a', 'poc']
    
    for f in tqdm(files, desc="Mining Historical Data"):
        try:
            df_chunk = pd.read_parquet(f)
            # Spatial Filter
            mask = (
                (df_chunk['lat'] >= box['lat_min']) & 
                (df_chunk['lat'] <= box['lat_max']) & 
                (df_chunk['lon'] >= box['lon_min']) & 
                (df_chunk['lon'] <= box['lon_max'])
            )
            region = df_chunk[mask]
            
            if not region.empty:
                stats = {'time': region['time'].iloc[0]}
                for var in target_vars:
                    # Rename internal names if necessary (e.g., 'chlor_a' -> 'chl')
                    col_name = var
                    if var not in region.columns:
                        # Try standard alternatives
                        if var == 'chlor_a' and 'chl' in region.columns: col_name = 'chl'
                    
                    if col_name in region.columns:
                        stats[f'{var}_mean'] = region[col_name].mean()
                        stats[f'{var}_max'] = region[col_name].max()
                        stats[f'{var}_min'] = region[col_name].min()
                        stats[f'{var}_std'] = region[col_name].std()
                
                history_data.append(stats)
            del df_chunk, region
        except: continue
        
    df = pd.DataFrame(history_data)
    df['time'] = pd.to_datetime(df['time'])
    if df['time'].dt.tz is not None:
        df['time'] = df['time'].dt.tz_localize(None)
        
    df['month'] = df['time'].dt.month
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    
    print(f"   -> Training Set: {len(df)} months of History.")
    return df

# ==========================================
# 2. XGBOOST ENGINE
# ==========================================
def train_and_predict():
    # A. Check Files
    # Auto-find the Forecast file (Stage I output)
    forecast_path = INPUT_FORECAST
    if not os.path.exists(forecast_path):
        # Search in outputs
        if os.path.exists(os.path.join("outputs", INPUT_FORECAST)):
            forecast_path = os.path.join("outputs", INPUT_FORECAST)
        elif os.path.exists(os.path.join("outputs/final_visuals_white", INPUT_FORECAST)):
            forecast_path = os.path.join("outputs/final_visuals_white", INPUT_FORECAST)
        else:
            print(f"[FAIL] Could not find Stage I output: {INPUT_FORECAST}")
            return

    # B. Load Data
    df_hist = load_training_history(DATA_DIR, REGION_BOX)
    df_future = pd.read_csv(forecast_path)
    df_future['time'] = pd.to_datetime(df_future['time'])
    
    # C. Feature Setup
    # Inputs (X): The Physical World
    features = [
        'sst_mean', 'sst_max', 'sst_min', 'sst_std',
        'par_mean', 'par_max', 'par_min', 'par_std',
        'kd_490_mean', 'kd_490_max', 'kd_490_min', 'kd_490_std',
        'month_sin', 'month_cos'
    ]
    
    # Targets (y): The Biological Response
    target_chl = 'chlor_a_mean'
    target_poc = 'poc_mean'
    
    # D. Train Model 1: Chlorophyll (Productivity)
    print("\n--- 2. Training Ecological Brain (XGBoost) ---")
    print("   -> Target: Chlorophyll-a (Ecosystem Productivity)")
    
    X_train = df_hist[features]
    y_train_chl = df_hist[target_chl]
    
    model_chl = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42)
    model_chl.fit(X_train, y_train_chl)
    
    # Predict Future Chl
    # Ensure future has same columns
    X_future = df_future[features] 
    pred_chl = model_chl.predict(X_future)
    
    # E. Train Model 2: POC (Biomass)
    print("   -> Target: POC (Total Biomass)")
    y_train_poc = df_hist[target_poc]
    
    model_poc = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42)
    model_poc.fit(X_train, y_train_poc)
    
    # Predict Future POC
    pred_poc = model_poc.predict(X_future)
    
    # F. Save Results
    df_future['pred_chlor_a_mean'] = pred_chl
    df_future['pred_poc_mean'] = pred_poc
    
    # Detect Bleaching Events (Anomaly Detection)
    # Logic: High Heat (SST > 30) AND Low Chlorophyll (Collapse)
    # We calculate a 'Stress Index'
    baseline_chl = df_hist['chlor_a_mean'].mean()
    df_future['bleaching_risk'] = np.where(
        (df_future['sst_max'] > 30.0) & (df_future['pred_chlor_a_mean'] < baseline_chl * 0.8), 
        1, 0
    )
    
    save_path = os.path.join(OUTPUT_DIR, "Stage2_Bio_Forecast.csv")
    df_future.to_csv(save_path, index=False)
    print(f"   -> Saved Biological Forecast: {save_path}")
    
    return df_hist, df_future

# ==========================================
# 3. VISUALIZATION (HOLLYWOOD V3)
# ==========================================
def visualize_impact(df_hist, df_pred):
    print("\n--- 3. Generating Impact Visuals ---")
    
    # Combine History and Future for continuous plot
    # Normalize column names for merging
    h_cols = ['time', 'chlor_a_mean', 'poc_mean']
    f_cols = ['time', 'pred_chlor_a_mean', 'pred_poc_mean']
    
    df_h_plot = df_hist[h_cols].copy()
    df_f_plot = df_pred[f_cols].rename(columns={'pred_chlor_a_mean': 'chlor_a_mean', 'pred_poc_mean': 'poc_mean'})
    
    full_df = pd.concat([df_h_plot, df_f_plot]).sort_values('time')
    
    # PLOT 1: Chlorophyll Collapse Check
    fig, ax = plt.subplots(figsize=(15, 7))
    
    # Plot History
    ax.plot(df_h_plot['time'], df_h_plot['chlor_a_mean'], color='gray', alpha=0.6, label='Historical Observation')
    
    # Plot Forecast
    ax.plot(df_f_plot['time'], df_f_plot['chlor_a_mean'], color=colors['chl'], linewidth=2.5, label='AI Forecast (2026-2035)')
    
    # Highlight Bleaching Risks
    risks = df_pred[df_pred['bleaching_risk'] == 1]
    if not risks.empty:
        ax.scatter(risks['time'], risks['pred_chlor_a_mean'], color='red', s=100, zorder=5, label='Predicted Bleaching Event')
    
    ax.set_title("Ecological Vitals: Chlorophyll-a Projection", fontsize=18, fontweight='bold')
    ax.set_ylabel("Concentration (mg/m³)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "1_Chlorophyll_Forecast.png"), dpi=300)
    plt.show()
    
    # PLOT 2: The "Death Spiral" (SST vs Chlorophyll)
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Scatter plot: Heat vs Biology
    # History
    ax.scatter(df_hist['sst_mean'], df_hist['chlor_a_mean'], color='gray', alpha=0.3, label='Historical Resilience')
    
    # Future
    sc = ax.scatter(df_pred['sst_mean'], df_pred['pred_chlor_a_mean'], 
                    c=df_pred['time'].dt.year, cmap='inferno', s=60, edgecolors='black', label='Future Trajectory')
    
    ax.set_xlabel("Sea Surface Temperature (°C)", fontweight='bold')
    ax.set_ylabel("Chlorophyll-a (Productivity)", fontweight='bold')
    ax.set_title("The Tipping Point: Heat Stress vs. Biological Collapse", fontsize=16, fontweight='bold')
    
    cbar = plt.colorbar(sc)
    cbar.set_label("Forecast Year")
    
    plt.savefig(os.path.join(OUTPUT_DIR, "2_Tipping_Point_Analysis.png"), dpi=300)
    plt.show()

if __name__ == "__main__":
    hist, future = train_and_predict()
    if hist is not None:
        visualize_impact(hist, future)